In [ ]:
import os
import json
import numpy as np
import torch
import random
import logging

from train import LSTMLMTrainer, VAETrainer

with open("Liu_Kheyer_Retrosynthesis_Data/reproduced/v4.3/config.json", "r", encoding="utf-8") as file:
    config = json.load(file)
    config = {
        param: value for _, params in config.items() for param, value in params.items()
    }
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(config["log_path"]),
                              logging.StreamHandler()])

config["cuda"] = config["cuda"] and torch.cuda.is_available()

logging.info(json.dumps(config, indent=4))

# fix seeds
torch.manual_seed(config["seed"])
torch.use_deterministic_algorithms(True)
np.random.seed(config["seed"])
random.seed(config["seed"])
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'

if config["cuda"]:
    logging.info("Using Cuda")
    torch.cuda.manual_seed(config["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

if config["mode"] == "train":
    if config["model"] == "LSTM":
        logging.info("training LSTM")
        trainer = LSTMLMTrainer(config=config)
    elif config["model"] == "VAE":
        logging.info("training VAE")
        trainer = VAETrainer(config=config)
    trainer.train()

## Transformer

In [1]:
import os
import json
import numpy as np
import torch
import random
import logging
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


with open("Liu_Kheyer_Retrosynthesis_Data/reproduced/v5/config.json", "r", encoding="utf-8") as file:
    config = json.load(file)
    config = {
        param: value for _, params in config.items() for param, value in params.items()
    }
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(config["log_path"]),
                              logging.StreamHandler()])

config["cuda"] = config["cuda"] and torch.cuda.is_available()

logging.info(json.dumps(config, indent=4))

# fix seeds
torch.manual_seed(config["seed"])
torch.use_deterministic_algorithms(True)
np.random.seed(config["seed"])
random.seed(config["seed"])
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'

if config["cuda"]:
    logging.info("Using Cuda")
    torch.cuda.manual_seed(config["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


2025-06-21 09:48:33,762 - INFO - {
    "save_path": "model_weights/synthesizer/reproduced/v5",
    "main_dir": "Liu_Kheyer_Retrosynthesis_Data",
    "data_ref_path": "train_ref_dataset.csv",
    "train_path": "Liu_Kheyer_Retrosynthesis_Data/train/train_targets_ids_200.data",
    "val_path": "Liu_Kheyer_Retrosynthesis_Data/validation/validation_targets_ids_200.data",
    "gene_path": "Liu_Kheyer_Retrosynthesis_Data/reproduced/v5/gene.data",
    "log_path": "Liu_Kheyer_Retrosynthesis_Data/reproduced/v5/train.log",
    "load_path": "",
    "model": "VAE",
    "mode": "train",
    "seed": 42,
    "n_gen_samples": 39579,
    "cuda": true,
    "vocab_size": 56,
    "batch_size": 180,
    "seq_len": 200,
    "epochs": 200,
    "lr": 1.0,
    "warmup_steps": 4000,
    "d_model": 512,
    "dim_feedforward": 2048,
    "nhead": 16,
    "num_layers": 6,
    "dropout": 0.1,
    "temperature": 1.0
}
2025-06-21 09:48:33,764 - INFO - Using Cuda


In [4]:
from models.Transformer.transformer import Transformer
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

src_vocab_size = 56
dim_feedforward = 2048
num_heads = 16
num_layers = 6
d_model = 512
dropout = 0.1
max_seq_len = 200
warmup_steps = 4000

transformer = Transformer(ntoken=src_vocab_size, ninp=d_model, nhead=num_heads, nhid=dim_feedforward, nlayers=num_layers,max_len = max_seq_len, dropout=dropout)

In [5]:
import sentencepiece as spm
from dataloader import DataIterator
import os
spm.SentencePieceTrainer.train(
    "--input=Liu_Kheyer_Retrosynthesis_Data/vocab2.txt --model_prefix=m  --user_defined_symbols=[BOS],[EOS],[PAD],. --vocab_size=56 --bos_id=-1 --eos_id=-1"
)
tokenizer = spm.SentencePieceProcessor()
tokenizer.load("m.model")
PAD_TOKEN = tokenizer.encode_as_ids("[PAD]")[1]
EOS_TOKEN = tokenizer.encode_as_ids("[EOS]")[1]
BOS_TOKEN = tokenizer.encode_as_ids("[BOS]")[1]
train_iter = DataIterator(
    data_file=os.path.join(config["train_path"]),
    batch_size=config["batch_size"],
    PAD_TOKEN=PAD_TOKEN,
)
eval_iter = DataIterator(
    data_file=os.path.join(config["val_path"]),
    batch_size=config["batch_size"],
    PAD_TOKEN=PAD_TOKEN,
)

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=Liu_Kheyer_Retrosynthesis_Data/vocab2.txt --model_prefix=m  --user_defined_symbols=[BOS],[EOS],[PAD],. --vocab_size=56 --bos_id=-1 --eos_id=-1
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: Liu_Kheyer_Retrosynthesis_Data/vocab2.txt
  input_format: 
  model_prefix: m
  model_type: UNIGRAM
  vocab_size: 56
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: [BOS]
  user_defined_symbols: [EOS]
  user_defined_symbols: [PAD]
  user_defined_symbols: .
  required_cha

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from eval import Evaluator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = transformer.to(device)
criterion = nn.NLLLoss(ignore_index=PAD_TOKEN,reduction="sum").to(device).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min((step + 1) ** -0.5, (step + 1) * (warmup_steps ** -1.5))
)

evaluator = Evaluator(config)
outf = "gene.data"
n_samples = 100
temperature = 1.0

for epoch in range(1, 101):
    model.train()
    total_train_loss = 0

    for data, target in train_iter:
        data = data.to(device)
        target = target.to(device)
        optimizer.zero_grad()

        output = model(data)
        log_probs = torch.log_softmax(output, dim=-1)
        loss = criterion(log_probs.view(-1, log_probs.size(-1)), target.view(-1))

        loss.backward()
        optimizer.step()
        scheduler.step()

        non_pad_mask = target.view(-1) != PAD_TOKEN
        num_valid_tokens = non_pad_mask.sum()
        loss = loss / num_valid_tokens 
        total_train_loss += loss.item()

    train_iter.reset()

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for data, target in eval_iter:
            data = data.to(device)
            target = target.to(device)
            output = model(data)
            log_probs = torch.log_softmax(output, dim=-1)
            val_loss = criterion(log_probs.view(-1, log_probs.size(-1)), target.view(-1))
            non_pad_mask = target.view(-1) != PAD_TOKEN
            num_valid_tokens = non_pad_mask.sum()
            val_loss = val_loss / num_valid_tokens 
            total_val_loss += val_loss.item()

    eval_iter.reset()

    avg_train_loss = total_train_loss / len(train_iter)
    avg_val_loss = total_val_loss / len(eval_iter)

    print(f"Epoch {epoch:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --------- Sample Generation ---------
    samples = []
    transformer.eval()
    with torch.no_grad():
        for _ in range(n_samples):
            input = torch.tensor([[BOS_TOKEN]], dtype=torch.long).to(device)
            generated = [BOS_TOKEN]

            for _ in range(1, max_seq_len):
                output = transformer(input, False)
                logits = output[-1].squeeze().div(temperature).exp().cpu()
                next_token = torch.multinomial(logits, 1)[0].item()

                generated.append(next_token)

                if next_token == EOS_TOKEN:
                    pad_len = max_seq_len - len(generated)
                    if pad_len > 0:
                        generated.extend([PAD_TOKEN] * pad_len)
                    break

                input = torch.cat(
                    [input, torch.tensor([[next_token]], dtype=torch.long).to(device)],
                    dim=0
                )

            if len(generated) < max_seq_len:
                generated.extend([PAD_TOKEN] * (max_seq_len - len(generated)))

            samples.append(generated)

    with open(outf, "w", encoding="utf-8") as fout:
        fout.writelines([" ".join(map(str, sample)) + "\n" for sample in samples])

    # --------- Metric Evaluation ---------
    print(f"Evaluating metrics after Epoch {epoch:03d}...")
    print(evaluator.generate_metrics_evaluation(outf))

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=Liu_Kheyer_Retrosynthesis_Data/vocab2.txt --model_prefix=m  --user_defined_symbols=[BOS],[EOS],[PAD],. --vocab_size=56 --bos_id=-1 --eos_id=-1
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: Liu_Kheyer_Retrosynthesis_Data/vocab2.txt
  input_format: 
  model_prefix: m
  model_type: UNIGRAM
  vocab_size: 56
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: [BOS]
  user_defined_symbols: [EOS]
  user_defined_symbols: [PAD]
  user_defined_symbols: .
  required_cha

Epoch 001 | Train Loss: 4.6336 | Val Loss: 4.2798


2025-06-21 03:03:51,383 - INFO - No Valid Reactions, exiting...


Evaluating metrics after Epoch 001...
JSS=0.0000, Sim=0.0000, StrSim=0.0000, Val=0.0000,
NoveltyPerc=0.0000, UniquePerc=0.0000, IntDiv=0.0000, OverallVal=0.0000,
 VS=0.0000, VS(q=0.1)=0.0000, VD(q=inf)=0.0000, AvgVSPerClass=0.0000
Epoch 002 | Train Loss: 3.8434 | Val Loss: 3.2543


2025-06-21 03:05:00,572 - INFO - No Valid Reactions, exiting...


Evaluating metrics after Epoch 002...
JSS=0.0000, Sim=0.0000, StrSim=0.0000, Val=0.0000,
NoveltyPerc=0.0000, UniquePerc=0.0000, IntDiv=0.0000, OverallVal=0.0000,
 VS=0.0000, VS(q=0.1)=0.0000, VD(q=inf)=0.0000, AvgVSPerClass=0.0000
Epoch 003 | Train Loss: 3.0951 | Val Loss: 2.6857


2025-06-21 03:06:17,116 - INFO - No Valid Reactions, exiting...


Evaluating metrics after Epoch 003...
JSS=0.0000, Sim=0.0000, StrSim=0.0000, Val=0.0000,
NoveltyPerc=0.0000, UniquePerc=0.0000, IntDiv=0.0000, OverallVal=0.0000,
 VS=0.0000, VS(q=0.1)=0.0000, VD(q=inf)=0.0000, AvgVSPerClass=0.0000
Epoch 004 | Train Loss: 2.6793 | Val Loss: 2.3662


2025-06-21 03:07:42,829 - INFO - No Valid Reactions, exiting...


Evaluating metrics after Epoch 004...
JSS=0.0000, Sim=0.0000, StrSim=0.0000, Val=0.0000,
NoveltyPerc=0.0000, UniquePerc=0.0000, IntDiv=0.0000, OverallVal=0.0000,
 VS=0.0000, VS(q=0.1)=0.0000, VD(q=inf)=0.0000, AvgVSPerClass=0.0000


In [ ]:
import os
import json
import numpy as np
import torch
import random
import logging

from train import LSTMLMTrainer, VAETrainer, TransformerTrainer
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

with open("Liu_Kheyer_Retrosynthesis_Data/reproduced/v5/config.json", "r", encoding="utf-8") as file:
    config = json.load(file)
    config = {
        param: value for _, params in config.items() for param, value in params.items()
    }
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(config["log_path"]),
                              logging.StreamHandler()])

config["cuda"] = config["cuda"] and torch.cuda.is_available()

logging.info(json.dumps(config, indent=4))

# fix seeds
torch.manual_seed(config["seed"])
torch.use_deterministic_algorithms(True)
np.random.seed(config["seed"])
random.seed(config["seed"])
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'

if config["cuda"]:
    logging.info("Using Cuda")
    torch.cuda.manual_seed(config["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

if config["mode"] == "train":
    if config["model"] == "LSTM":
        logging.info("training LSTM")
        trainer = LSTMLMTrainer(config=config)
    elif config["model"] == "VAE":
        logging.info("training VAE")
        trainer = VAETrainer(config=config)
    elif config["model"] == "Transformer":
        logging.info("training Transformer")
        trainer = TransformerTrainer(config=config)
    trainer.train()

In [ ]:
import os
import json
import numpy as np
import torch
import random
import logging

from train import LSTMLMTrainer, VAETrainer, TransformerTrainer
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

with open("Liu_Kheyer_Retrosynthesis_Data/reproduced/v5.2/config.json", "r", encoding="utf-8") as file:
    config = json.load(file)
    config = {
        param: value for _, params in config.items() for param, value in params.items()
    }
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(config["log_path"]),
                              logging.StreamHandler()])

config["cuda"] = config["cuda"] and torch.cuda.is_available()

logging.info(json.dumps(config, indent=4))

# fix seeds
torch.manual_seed(config["seed"])
torch.use_deterministic_algorithms(True)
np.random.seed(config["seed"])
random.seed(config["seed"])
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'

if config["cuda"]:
    logging.info("Using Cuda")
    torch.cuda.manual_seed(config["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

if config["mode"] == "train":
    if config["model"] == "LSTM":
        logging.info("training LSTM")
        trainer = LSTMLMTrainer(config=config)
    elif config["model"] == "VAE":
        logging.info("training VAE")
        trainer = VAETrainer(config=config)
    elif config["model"] == "Transformer":
        logging.info("training Transformer")
        trainer = TransformerTrainer(config=config)
    trainer.train()

2025-06-22 19:31:58,755 - INFO - {
    "save_path": "model_weights/synthesizer/reproduced/v5.2",
    "main_dir": "Liu_Kheyer_Retrosynthesis_Data",
    "data_ref_path": "train_ref_dataset.csv",
    "train_path": "Liu_Kheyer_Retrosynthesis_Data/train/train_targets_ids_200.data",
    "val_path": "Liu_Kheyer_Retrosynthesis_Data/validation/validation_targets_ids_200.data",
    "gene_path": "Liu_Kheyer_Retrosynthesis_Data/reproduced/v5.2/gene.data",
    "log_path": "Liu_Kheyer_Retrosynthesis_Data/reproduced/v5.2/train.log",
    "load_path": "",
    "model": "Transformer",
    "mode": "train",
    "seed": 42,
    "n_gen_samples": 100,
    "cuda": true,
    "vocab_size": 56,
    "batch_size": 180,
    "seq_len": 200,
    "epochs": 200,
    "lr": 1,
    "warmup_steps": 4000,
    "d_model": 512,
    "dim_feedforward": 2048,
    "nhead": 16,
    "num_layers": 6,
    "dropout": 0.5,
    "temperature": 1.0
}
2025-06-22 19:31:58,758 - INFO - Using Cuda
2025-06-22 19:31:58,759 - INFO - training Trans

In [ ]:
import os
import json
import numpy as np
import torch
import random
import logging

from train import LSTMLMTrainer, VAETrainer, TransformerTrainer
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

with open("Liu_Kheyer_Retrosynthesis_Data/reproduced/v5.3/config.json", "r", encoding="utf-8") as file:
    config = json.load(file)
    config = {
        param: value for _, params in config.items() for param, value in params.items()
    }
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(config["log_path"]),
                              logging.StreamHandler()])

config["cuda"] = config["cuda"] and torch.cuda.is_available()

logging.info(json.dumps(config, indent=4))

# fix seeds
torch.manual_seed(config["seed"])
torch.use_deterministic_algorithms(True)
np.random.seed(config["seed"])
random.seed(config["seed"])
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'

if config["cuda"]:
    logging.info("Using Cuda")
    torch.cuda.manual_seed(config["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

if config["mode"] == "train":
    if config["model"] == "LSTM":
        logging.info("training LSTM")
        trainer = LSTMLMTrainer(config=config)
    elif config["model"] == "VAE":
        logging.info("training VAE")
        trainer = VAETrainer(config=config)
    elif config["model"] == "Transformer":
        logging.info("training Transformer")
        trainer = TransformerTrainer(config=config)
    trainer.train()

2025-06-23 11:42:33,165 - INFO - {
    "save_path": "model_weights/synthesizer/reproduced/v5.3",
    "main_dir": "Liu_Kheyer_Retrosynthesis_Data",
    "data_ref_path": "train_ref_dataset.csv",
    "train_path": "Liu_Kheyer_Retrosynthesis_Data/train/train_targets_ids_200.data",
    "val_path": "Liu_Kheyer_Retrosynthesis_Data/validation/validation_targets_ids_200.data",
    "gene_path": "Liu_Kheyer_Retrosynthesis_Data/reproduced/v5.3/gene.data",
    "log_path": "Liu_Kheyer_Retrosynthesis_Data/reproduced/v5.3/train.log",
    "load_path": "",
    "model": "Transformer",
    "mode": "train",
    "seed": 42,
    "n_gen_samples": 100,
    "cuda": true,
    "vocab_size": 56,
    "batch_size": 180,
    "seq_len": 200,
    "epochs": 200,
    "lr": 1,
    "warmup_steps": 4000,
    "d_model": 512,
    "dim_feedforward": 2048,
    "nhead": 16,
    "num_layers": 6,
    "dropout": 0.5,
    "temperature": 1.0,
    "label_smoothing": 0.1
}
2025-06-23 11:42:33,166 - INFO - Using Cuda
2025-06-23 11:42:33

KeyboardInterrupt: 